## Adaptive Boosting

Adaptive Boosting, also known as AdaBoost, is an ensemble learning technique that combines multiple weak classifiers to create a strong classifier. The main idea behind AdaBoost is to iteratively train weak classifiers on the data, giving more weight to misclassified instances in each subsequent iteration. This way, the algorithm focuses on the difficult cases and improves the overall performance of the model.

Unlike Gradient Boosting, which optimizes a loss function, AdaBoost focuses on adjusting the weights of the training instances to improve the performance of the weak classifiers. The final prediction is made by combining the predictions of all weak classifiers, weighted by their respective performance.

The following formula is used to calculate the weight of each weak classifier based on its error rate:

$$\alpha_t = \frac{1}{2} \ln\left(\frac{1-\epsilon}{\epsilon}\right)$$

Where:
- $\alpha_t$ is the weight of the weak classifier at iteration $t$.
- $\epsilon$ is the error rate of the weak classifier, calculated as the proportion of misclassified instances.

In [22]:
import pandas as pd

data = {"Person": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 
        "Red": [1, 1, 1, 1, 0, 1, 1, 0, 0, 1], 
        "Yellow": [1, 0, 1, 1, 1, 0, 1, 1, 0, 1], 
        "Green": [1, 1, 0, 1, 1, 0, 0, 1, 1, 1], 
        "Blue": [1, 1, 1, 0, 1, 1, 0, 0, 1, 1], 
        "Target": ["Not Colorblind", "Not Colorblind", "Not Colorblind", 
                    "Not Colorblind", "Not Colorblind", "Colorblind", 
                    "Colorblind", "Colorblind", "Colorblind", "Not Colorblind"]}

df = pd.DataFrame(data)

# We use Label Encoding for demonstration only
for i in df.columns:
  unique_columns = df[i].unique()
  df[i] = df[i].map({unique_columns[value]: value for value in range(len(unique_columns))})

df['Weight'] = 1 / len(df)
print(df)

   Person  Red  Yellow  Green  Blue  Target  Weight
0       0    0       0      0     0       0     0.1
1       1    0       1      0     0       0     0.1
2       2    0       0      1     0       0     0.1
3       3    0       0      0     1       0     0.1
4       4    1       0      0     0       0     0.1
5       5    0       1      1     0       1     0.1
6       6    0       0      1     1       1     0.1
7       7    1       0      0     1       1     0.1
8       8    1       1      0     0       1     0.1
9       9    0       0      0     0       0     0.1


In [14]:
df['Predicted Label'] = [0, 1, 0, 0, 0, 0, 1, 1, 1, 1]
df['Actual Label'] = df['Target']

misclassifed_samples = len(df[df['Predicted Label'] != df['Actual Label']])
eps = 0.1 * misclassifed_samples
print("Epsilon:", eps)

Epsilon: 0.30000000000000004


In [15]:
import numpy as np

learn_weights = 0.5 * np.log((1 - eps) / eps)
print("Learner Weights:", learn_weights)

Learner Weights: 0.4236489301936017


In [16]:
# Misclassified Samples
df.loc[df['Predicted Label'] != df['Actual Label'], 'Weight'] = 0.1 * np.exp(learn_weights)
# Correctly classified Samples
df.loc[df['Predicted Label'] == df['Actual Label'], 'Weight'] = 0.1 * np.exp(-learn_weights)
print(df[['Actual Label', 'Predicted Label', 'Weight']])
# Compute total weights
total_weight = df['Weight'].sum()
print("Total Weight:", total_weight)

   Actual Label  Predicted Label    Weight
0             0                0  0.065465
1             0                1  0.152753
2             0                0  0.065465
3             0                0  0.065465
4             0                0  0.065465
5             1                0  0.152753
6             1                1  0.065465
7             1                1  0.065465
8             1                1  0.065465
9             0                1  0.152753
Total Weight: 0.9165151389911681


In [17]:
# Normalize the weight
df.loc[df['Predicted Label'] != df['Actual Label'], 'Weight'] = df['Weight'] / total_weight
df.loc[df['Predicted Label'] == df['Actual Label'], 'Weight'] = df['Weight'] / total_weight
print(df[['Actual Label', 'Predicted Label', 'Weight']])
total_weight = df['Weight'].sum()
print("Total Weight:", total_weight)

   Actual Label  Predicted Label    Weight
0             0                0  0.071429
1             0                1  0.166667
2             0                0  0.071429
3             0                0  0.071429
4             0                0  0.071429
5             1                0  0.166667
6             1                1  0.071429
7             1                1  0.071429
8             1                1  0.071429
9             0                1  0.166667
Total Weight: 0.9999999999999999


In [18]:
# The only difference is the predicted label
df['Predicted Label2'] = [0, 0, 0, 0, 1, 1, 1, 1, 0, 0]

misclassifed_samples2 = len(df[df['Predicted Label2'] != df['Actual Label']])
eps2 = 0.1 * misclassifed_samples2
print("Epsilon 2:", eps2)

Epsilon 2: 0.2


In [19]:
import numpy as np

learn_weights2 = 0.5 * np.log((1 - eps2) / eps2)
print("Learner Weights 2:", learn_weights2)

Learner Weights 2: 0.6931471805599453


In [20]:
# Misclassified Samples
df.loc[df['Predicted Label2'] != df['Actual Label'], 'Weight'] = df['Weight'] * np.exp(learn_weights2)
# Classified samples 
df.loc[df['Predicted Label2'] == df['Actual Label'], 'Weight'] = df['Weight'] * np.exp(-learn_weights2)

print(df[['Actual Label', 'Predicted Label2', 'Weight']])

total_weight = df['Weight'].sum()
print("Total Weight:", total_weight)

   Actual Label  Predicted Label2    Weight
0             0                 0  0.035714
1             0                 0  0.083333
2             0                 0  0.035714
3             0                 0  0.035714
4             0                 1  0.142857
5             1                 1  0.083333
6             1                 1  0.035714
7             1                 1  0.035714
8             1                 0  0.142857
9             0                 0  0.083333
Total Weight: 0.7142857142857144


In [21]:
# Normalize the weight
df.loc[df['Predicted Label2'] != df['Actual Label'], 'Weight'] = df['Weight'] / total_weight
df.loc[df['Predicted Label2'] == df['Actual Label'], 'Weight'] = df['Weight'] / total_weight
print("\nNormalized:\n", df[['Actual Label', 'Predicted Label2', 'Weight']])
total_weight = df['Weight'].sum()
print("Total Weight:", total_weight)


Normalized:
    Actual Label  Predicted Label2    Weight
0             0                 0  0.050000
1             0                 0  0.116667
2             0                 0  0.050000
3             0                 0  0.050000
4             0                 1  0.200000
5             1                 1  0.116667
6             1                 1  0.050000
7             1                 1  0.050000
8             1                 0  0.200000
9             0                 0  0.116667
Total Weight: 0.9999999999999998
